# TruthLens audio detector on a Colab GPU

Downloads ASVspoof 2019 LA, trains LCNN with LFCC and the self-supervised models, then evaluates them with min t-DCF and exports LCNN to ONNX.

## Setup

The notebook clones the repository from GitHub, so push your local commits first. Choose a GPU runtime. Everything that must survive a disconnect is written to Google Drive.

In [ ]:
REPO = 'https://github.com/Shashank17singh/TruthLens.git'
BRANCH = 'main'
import os
if not os.path.exists('/content/TruthLens'):
    !git clone -q --branch {BRANCH} {REPO} /content/TruthLens
%cd /content/TruthLens
!git pull -q
assert os.path.exists('ml/audio/train.py'), 'This branch does not have ml/audio/train.py yet. Push your commits, then run this cell again.'

In [ ]:
!pip install -q -r ml/requirements.txt
import torch
assert torch.cuda.is_available(), 'Choose a GPU runtime: Runtime, Change runtime type.'
print(torch.cuda.get_device_name(0))

If pip asks you to restart the session, restart it and run the two cells above again. The clone is kept.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/truthlens'
os.makedirs(DRIVE, exist_ok=True)

## Data

ASVspoof 2019 LA is 7.6 GB. The server drops long transfers, so the download resumes until the file is complete, and every member's CRC is checked before use.

In [ ]:
RAW = 'data/raw/asvspoof2019_la'
SIZE = 7640952520
URL = 'https://datashare.ed.ac.uk/bitstream/handle/10283/3336/LA.zip'
os.makedirs(RAW, exist_ok=True)
for attempt in range(200):
    if os.path.exists(f'{RAW}/LA.zip') and os.path.getsize(f'{RAW}/LA.zip') >= SIZE:
        break
    !curl -sS -L -C - --max-time 1800 -o {RAW}/LA.zip {URL}
print(os.path.getsize(f'{RAW}/LA.zip'), 'bytes')

In [ ]:
import zipfile
with zipfile.ZipFile(f'{RAW}/LA.zip') as bundle:
    bad = bundle.testzip()
assert bad is None, f'Corrupt member {bad}. Delete LA.zip and download again.'
!python -m ml.common.prepare asvspoof2019_la --workers 4

## Train

LCNN with LFCC is the model the service deploys. The self-supervised models need the GPU. Each command resumes after a disconnect.

In [ ]:
CKPT = f'{DRIVE}/checkpoints/audio'
!python -m ml.audio.train model=lcnn feature=lfcc checkpoint_dir={CKPT} tracker=none num_workers=2

In [ ]:
!python -m ml.audio.train model=wav2vec2 checkpoint_dir={CKPT} tracker=none num_workers=2 steps_per_epoch=2000

In [ ]:
!python -m ml.audio.train model=wavlm checkpoint_dir={CKPT} tracker=none num_workers=2 steps_per_epoch=2000

## Evaluate and export

With the official ASV scores, the report adds min t-DCF for the evaluation set.

In [ ]:
import glob
ASV = f'{RAW}/LA/ASVspoof2019_LA_asv_scores/ASVspoof2019.LA.asv.eval.gi.trl.scores.txt'
for best in sorted(glob.glob(f'{CKPT}/*/*-best.pt')):
    run = os.path.basename(os.path.dirname(best))
    !python -m ml.audio.evaluate {best} --asv-scores {ASV} --output {DRIVE}/results/audio/{run}

In [ ]:
LCNN = glob.glob(f'{CKPT}/audio-lcnn-lfcc-*/*-best.pt')[0]
!python -m ml.audio.export {LCNN} --output-dir {DRIVE}/onnx

Copy the results folders from Drive into `ml/eval/results/audio/` and the ONNX file into `checkpoints/onnx/` in your local clone.